# Backtesting

Backtesting er å teste en strategi på historiske data for å se hvordan den ville ha fungert tidligere. Det brukes ofte i investering for å vurdere om en strategi kan være verdt å bruke, uten at det gir noen garanti for fremtidige resultater.

## Importering og Henting av data

In [61]:
# Importing
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import os

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.timeframe import TimeFrame
from alpaca.data.requests import StockBarsRequest
from alpaca.trading.client import TradingClient

In [62]:
# Loading dotenv
load_dotenv()

KEY = os.getenv("key")
SECRET = os.getenv("secret")

In [63]:
# Henting av data
client = StockHistoricalDataClient(KEY, SECRET)

df = client.get_stock_bars(StockBarsRequest(
    symbol_or_symbols="AAPL",
    timeframe=TimeFrame.Day,
    start = "2026-01-01"
)).df

df.describe()

,open,high,low,close,volume,trade_count,vwap
count,116.000000,116.000000,116.000000,116.000000,1.160000e+02,1.160000e+02,116.000000
mean,272.328819,275.322308,269.465963,272.371810,4.811077e+07,7.230678e+05,272.392196
std,19.109958,19.479529,19.092432,19.189208,1.477960e+07,1.620475e+05,19.285570
min,247.320000,249.199900,243.420000,246.630000,2.637259e+07,4.825740e+05,246.869425
25%,258.037500,260.202500,255.517500,258.692500,3.889984e+07,6.161120e+05,257.976611
50%,266.880000,269.960000,262.670000,266.175000,4.466828e+07,7.018710e+05,266.009827
75%,289.455000,292.567500,286.180000,288.270000,5.247779e+07,7.779930e+05,289.507325
max,314.175000,317.400000,309.650000,315.200000,9.244341e+07,1.333467e+06,313.143117


In [64]:
# Henting av brukerdata
trader = TradingClient(KEY, SECRET)

## Beregning av QTY - Position-Sizing

$$ Qty = \frac{Equity \times Risk Pct}{2 \times ATR} $$


## Definering av en stategy
**F.eks: SMA-Crossover-Strategy**

In [65]:
from typing import Tuple
from alpaca.trading.enums import OrderSide

def sma_strategy(equity: int, df: pd.DataFrame, risk_pct: float = 0.01) -> Tuple[OrderSide, int, float]:
    """
    En trendfølgende SMA-strategi med ATR-basert posisjonsstørrelse og pyramidering.

    Args:
        equity (int): En Integer som forteller egenkapitalen til brukeren.
        df (pd.DataFrame): Den historiske dataen til en akjse konvertert til en pandas dataframe.
        risk_pct (float): En faktor som forteller hvor mye av egenkapitalen vi er villig til å risikere.

    Retruns Tuple[OrderSide, int]:
        OrderSide: Et signal som forteller om brukeren burde kjøpe, selge eller ikke sende ut en ordre av aksjen.
        int: Antall aksjer brukeren burde fylle ordren sin med.
        float: Stop_Loss
    """
    # Buy or Sell
    sma50 = df["close"].rolling(50).mean().iloc[-1]
    sma200 = df["close"].rolling(200).mean().iloc[-1]

    # QTY using ATR
    high_low = df["high"] - df["low"]
    high_close = abs(df["high"] - df["close"].shift())
    low_close = abs(df["low"] - df["close"].shift())

    tr = pd.concat([high_close, high_low, low_close], axis=1).max(axis=1)
    atr = tr.rolling(14).mean().iloc[-1]
    
    risk_per_share = 2*atr
    risk_amount = equity * risk_pct

    qty = int(risk_amount / risk_per_share)
    stop_loss = df["close"].iloc[-1] - risk_per_share

    # Logic
    if qty == 0:
        return None, 0, stop_loss

    if sma50 > sma200:
        # Har du råd?
        max_qty = int(equity / df["close"].iloc[-1])
        qty = min(qty, max_qty)
        
        return OrderSide.BUY, qty, stop_loss
    
    elif sma50 < sma200:
        return OrderSide.SELL, qty, stop_loss
    
    else:
        return None, 0, stop_loss

## Backtesting

In [66]:
start_cash = 100000
equity = start_cash # Eksempel på hvor mye egenkapital vi vil backteste på
positions = 0
entry_price = None

trades = []
equity_curve = []

backtest_days = client.get_stock_bars(StockBarsRequest(
    symbol_or_symbols="AAPL",
    timeframe=TimeFrame.Day,
    start = "2020-01-01",
    end = "2025-01-01"
)).df


for day_i in range(200, len(backtest_days)): # 200 fordi vi må tenke på at sma200 trenger 200 dager som forberedelse.
    row = backtest_days.iloc[day_i]
    
    print("-"*30)
    # Bruk kun data frem til dagen før for å unngå look-ahead bias.
    signal, qty, suggested_stop = sma_strategy(equity, backtest_days.iloc[:day_i])
    print(f"{day_i}: {signal}, {qty}")

    # Buying with no positions
    if positions == 0 and signal == OrderSide.BUY and qty > 0:
        print("Buying")
        positions = qty
        entry_price = row["open"]
        stop_loss = suggested_stop
        equity -= qty * entry_price

        trades.append({
            "date": backtest_days.index[day_i],
            "day_i": day_i,
            "pnl": 0,
            "side": OrderSide.BUY,
            "qty": qty,
        })

    elif positions > 0:

        # Stop Loss
        if row["low"] <= stop_loss:
            print("STOPP LOSS Crossed, Selling")

            sold_qty = positions    # for log-føring
            exit_price = min(row["open"], stop_loss)

            equity += sold_qty * exit_price
            pnl = (exit_price - entry_price) * positions
            positions = 0
            entry_price = None
            stop_loss = None

            trades.append({
                "date": backtest_days.index[day_i],
                "day_i": day_i,
                "pnl": pnl,
                "side": OrderSide.SELL,
                "qty": sold_qty,
            })


        # Selling
        elif signal == OrderSide.SELL:
            print("Selling")

            qty = min(positions, qty)
            pnl = (row["open"] - entry_price) * qty
            equity += qty * row["open"]
            positions -= qty

            trades.append({
                "date": backtest_days.index[day_i],
                "day_i": day_i,
                "pnl": pnl,
                "side": OrderSide.SELL,
                "qty": qty,
            })

        # Buying
        elif signal == OrderSide.BUY:
            print("Buying")

            entry_price = (entry_price * positions + row["open"] * qty) / (positions + qty) # Gjennomsnittlig entry price.
            positions += qty
            stop_loss = max(stop_loss, suggested_stop) # Velger strammeste stop-loss
            equity -= qty * row["open"]

            trades.append({
                "date": backtest_days.index[day_i],
                "day_i": day_i,
                "pnl": 0,
                "side": OrderSide.BUY,
                "qty": qty,
            })

        # Holding
        else:
            print("Holding")

    portfolio_value = equity + positions * row["close"]
    equity_curve.append(portfolio_value)

# Selge alle positions
if positions > 0:
    final_price = backtest_days.iloc[-1]["close"]

    equity += positions * final_price

    trades.append({
        "date": backtest_days.index[-1],
        "day_i": len(backtest_days) - 1,
        "pnl": (final_price - entry_price) * positions,
        "side": OrderSide.SELL,
        "qty": positions,
    })

    positions = 0



------------------------------
200: OrderSide.SELL, 141
------------------------------
201: OrderSide.SELL, 142
------------------------------
202: OrderSide.SELL, 133
------------------------------
203: OrderSide.SELL, 134
------------------------------
204: OrderSide.SELL, 133
------------------------------
205: OrderSide.SELL, 136
------------------------------
206: OrderSide.SELL, 140
------------------------------
207: OrderSide.SELL, 142
------------------------------
208: OrderSide.SELL, 141
------------------------------
209: OrderSide.SELL, 131
------------------------------
210: OrderSide.SELL, 123
------------------------------
211: OrderSide.SELL, 124
------------------------------
212: OrderSide.SELL, 129
------------------------------
213: OrderSide.SELL, 131
------------------------------
214: OrderSide.SELL, 126
------------------------------
215: OrderSide.SELL, 122
------------------------------
216: OrderSide.SELL, 125
------------------------------
217: OrderSide.SE

## Definisjoner på ulike resultater

### Total Return (%)

Total Return viser den totale prosentvise avkastningen strategien har generert gjennom backtest-perioden:

$$
Total\ Return=
\frac{
Final\ Value-Initial\ Value
}
{
Initial\ Value
}
\times100
$$

<br>

### Final Value

Final Value representerer sluttverdien av porteføljen etter at backtesten er ferdig:

$$
Final\ Value=
Initial\ Value
+
Profit
-
Costs
$$

<br>

### Sharpe Ratio

Sharpe Ratio måler risikojustert avkastning ved å sammenligne avkastning med volatilitet.

Høyere Sharpe Ratio indikerer generelt bedre risikojustert ytelse:

$$
Sharpe=
\frac{
R_p-R_f
}
{
\sigma_p
}
$$

<br>

| Sharpe Ratio  | Vurdering                                                 |
| ------------- | --------------------------------------------------------- |
| **< 0**       | Dårlig – du fikk dårligere avkastning enn risikofri rente |
| **0 – 0.5**   | Svak                                                      |
| **0.5 – 1.0** | Under middels                                             |
| **1.0 – 1.5** | Grei / akseptabel                                         |
| **1.5 – 2.0** | Bra                                                       |
| **2.0 – 3.0** | Veldig bra                                                |
| **> 3.0**     | Eksepsjonell (og verdt å dobbeltsjekke for overfitting)   |

<br>

### Drawdown

Drawdown beskriver hvor mye porteføljen har falt fra en tidligere toppverdi:

$$
Drawdown=
\frac{
Peak-Current
}
{
Peak
}
\times100
$$

<br>

### Maximum Drawdown (Max DD)

Maximum Drawdown er det største fallet porteføljen opplevde gjennom hele backtest-perioden:

$$
Max\ Drawdown=
\max(Drawdown)
$$

<br>

### Win Rate (%)

Win Rate viser hvor stor andel av handler som endte med gevinst:

$$
Win\ Rate=
\frac{
Winning\ Trades
}
{
Total\ Trades
}
\times100
$$

<br>

### Benchmark

En benchmark er en referanse som brukes for å evaluere strategiens ytelse.

Vanlige benchmarks inkluderer:

- S&P 500
- Nasdaq 100
- MSCI World

<br>

### Benchmarking

Benchmarking er prosessen med å sammenligne strategiens resultater mot en benchmark.

Målet er å avgjøre om strategien faktisk tilfører verdi utover markedet.

<br>

### Alpha

Alpha beskriver hvor mye en strategi har prestert bedre eller dårligere enn benchmarken:

$$
Alpha=
Strategy\ Return
-
Benchmark\ Return
$$

<br>


## Resultater

In [67]:
# Resultat
trades_df = pd.DataFrame(trades)

print(f"PnL: {trades_df['pnl'].sum()}")
print(f"Final Value: {equity}")

total_return = (equity - start_cash) / start_cash
print(f"Total Return: {total_return:.2%}")

closed_trades = [
    t for t in trades
    if t["side"] == OrderSide.SELL
]

wins = [
    t for t in closed_trades
    if t["pnl"] > 0
]

win_rate = len(wins) / len(closed_trades)
print(f"Win Rate: {win_rate:.2%}")

peak = equity_curve[0]
max_drawdown = 0

for value in equity_curve:
    peak = max(peak, value)

    dd = (value - peak) / peak

    max_drawdown = min(max_drawdown, dd)


print(f"Max DD: {max_drawdown:.2%}")

returns = []

for i in range(1, len(equity_curve)):
    r = (
        equity_curve[i]
        / equity_curve[i-1]
        - 1
    )

    returns.append(r)

rf = 0      # risikofri rente

sharpe = (
    np.mean(returns) - rf/252
) / np.std(returns)

sharpe *= np.sqrt(252)

print(f"Sharpe Ratio: {sharpe:.2f}")

benchmark_return = (
    backtest_days.iloc[-1]["close"]
    /
    backtest_days.iloc[200]["close"]
    - 1
)

alpha = (
    total_return
    - benchmark_return
)

print(f"Alpha: {alpha:2%}")

PnL: 9162.066528571504
Final Value: 109162.0665285714
Total Return: 9.16%
Win Rate: 56.76%
Max DD: -30.36%
Sharpe Ratio: 0.21
Alpha: -101.239547%


- PnL (Profit and Loss): Hvor mange dollar strategien tjente totalt.
- Final Value: Hvor mye porteføljen var verdt ved slutten av backtesten.
- Total Return: Prosentvis avkastning fra startkapital til sluttkapital.
- Win Rate: Andelen avsluttede handler som endte med gevinst.
- Max Drawdown: Det største fallet fra en tidligere topp i porteføljeverdien.
- Sharpe Ratio: Hvor mye risikojustert avkastning strategien ga.
- Alpha: Hvor mye bedre eller dårligere strategien gjorde det sammenlignet med benchmarken.